In [2]:
import os
import sys

# Get the workspace root directory (one level up from notebook directory)
workspace_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(workspace_root)
from llm_based_annotation.utils_extraction import extract_body, tokenize, clean_tokens, chunk_tokens, extract_few_shot_examples

project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"

# First Selection

In [12]:
# ---------- Define Hyperparameters ----------
fs_min_tokens = 100



In [ ]:
fs_filename = "1989CanLII1415ONCA_annotated_GL_tech" #"2021QCCA1675_annotated_EG_tech" #"1997CanLII16226_ONCA_annotated_EG_tech" #"2019SCC65_annotated_EG_tech_corrected"
fs_anno = "GL"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}.html"


   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\GL\1989CanLII1415ONCA_annotated_GL_tech.html


In [20]:
fs_filename = "2019SCC65_annotated_EG_revRL" #"2001CanLII21117QCTDP_annotated_GL_tech" #"2024NBKB203_annotated_VP"
fs_html_path = fr"{project_root}\data\final\Annotated\{fs_filename}.html"

In [21]:
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")

   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\final\Annotated\2019SCC65_annotated_EG_revRL.html


In [22]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk = chunk_tokens(normalized_cleaned_tokens, min_tokens=fs_min_tokens, stop_bookmark_separation=False)

   ✓ Chunked tokens into 1237 chunks (>= 100 tokens each)


In [23]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources", "title", "citation", "fragment", "source", "authors"]
}

In [24]:
# ---------- Create few-shot examples ----------

# Pass chunks as a list (the function expects an iterable of chunks)
few_shot_examples = extract_few_shot_examples(token_chunk, 
                                              label_config)


print(f"   ✓ Selected {len(few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 1237 few-shot examples from chunks
   ✓ Selected 1237 few-shot examples for processing.


In [25]:
few_shot_examples

[(' SUPREME COURT OF CANADA Citation: Canada (Minister of Citizenship and Immigration) v. Vavilov, 2019 SCC 65, [2019] 4 S.C.R. 653 Appeal Heard: December 4, 5, 6, 2018 Judgment Rendered: December 19, 2019 Docket: 37748 Between:\nMinister',
  ' SUPREME COURT OF CANADA Citation: <decision><title>Canada (Minister of Citizenship and Immigration) v. Vavilov</title>,<citation> 2019 SCC 65</citation>, <citation>[2019] 4 S.C.R. 653</citation></decision> Appeal Heard: December 4, 5, 6, 2018 Judgment Rendered: December 19, 2019 Docket: 37748 Between:\nMinister'),
 (' of Citizenship and Immigration\nAppellant and Alexander Vavilov\nRespondent - and - Attorney General of Ontario, Attorney General of Quebec, Attorney General of British Columbia, Attorney General of Saskatchewan, Canadian Council for Refugees, Advocacy Centre for Tenants Ontario - Tenant Duty Counsel Program, Ontario Securities Commission, British',
  ' of Citizenship and Immigration\nAppellant and Alexander Vavilov\nRespondent - a

In [26]:
import json

# ---------- Save examples to JSON file ----------

# Format examples for the selection tool
formatted_examples = []
for ex in few_shot_examples:
    formatted_examples.append({
        "example": {"input": ex[0], "output": ex[1]},
        "comments": "",
        "selected": False,
        "notation": 0
    })

# Define output path
output_json_path = fr"{project_root}\few_shot_selection_tool\original\few_shot_examples_{fs_filename}.json"

# Save to JSON file
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(formatted_examples, f, indent=2, ensure_ascii=False)

print(f"   ✓ Saved {len(formatted_examples)} examples to: {output_json_path}")
print(f"   → Open few_shot_selection_tool/index.html and load this file to review and select examples")

   ✓ Saved 1237 examples to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\original\few_shot_examples_2019SCC65_annotated_EG_revRL.json
   → Open few_shot_selection_tool/index.html and load this file to review and select examples


# Second Selection

In [3]:
import json
import os
import glob
import re

# ---------- Configuration ----------
first_selected_folder = fr"{project_root}\few_shot_selection_tool\first_selected"
output_combined_path = fr"{project_root}\few_shot_selection_tool\first_selected\combined_selected_examples.json"

# ---------- Label detection function ----------
def extract_meta_flags(output_text):
    """
    Analyze the output text to detect which labels are present.
    Returns a dict with boolean flags for each label type.
    """
    meta = {
        "legislation": False,
        "legislation-title": False,
        "legislation-fragment": False,
        "legislation-citation": False,
        "decision": False,
        "decision-title": False,
        "decision-fragment": False,
        "decision-citation": False,
        "secondary sources": False,
        "secondary sources-title": False,
        "secondary sources-fragment": False,
        "secondary sources-source": False,
        "secondary sources-authors": False
    }
    
    # Check for main labels
    if "<legislation>" in output_text:
        meta["legislation"] = True
    if "<decision>" in output_text:
        meta["decision"] = True
    if "<secondary sources>" in output_text:
        meta["secondary sources"] = True
    
    # Check for legislation sub-labels
    if re.search(r'<legislation>.*?<title>.*?</title>.*?</legislation>', output_text, re.DOTALL):
        meta["legislation-title"] = True
    if re.search(r'<legislation>.*?<fragment>.*?</fragment>.*?</legislation>', output_text, re.DOTALL):
        meta["legislation-fragment"] = True
    if re.search(r'<legislation>.*?<citation>.*?</citation>.*?</legislation>', output_text, re.DOTALL):
        meta["legislation-citation"] = True
    
    # Check for decision sub-labels
    if re.search(r'<decision>.*?<title>.*?</title>.*?</decision>', output_text, re.DOTALL):
        meta["decision-title"] = True
    if re.search(r'<decision>.*?<fragment>.*?</fragment>.*?</decision>', output_text, re.DOTALL):
        meta["decision-fragment"] = True
    if re.search(r'<decision>.*?<citation>.*?</citation>.*?</decision>', output_text, re.DOTALL):
        meta["decision-citation"] = True
    
    # Check for secondary sources sub-labels
    if re.search(r'<secondary sources>.*?<title>.*?</title>.*?</secondary sources>', output_text, re.DOTALL):
        meta["secondary sources-title"] = True
    if re.search(r'<secondary sources>.*?<fragment>.*?</fragment>.*?</secondary sources>', output_text, re.DOTALL):
        meta["secondary sources-fragment"] = True
    if re.search(r'<secondary sources>.*?<source>.*?</source>.*?</secondary sources>', output_text, re.DOTALL):
        meta["secondary sources-source"] = True
    if re.search(r'<secondary sources>.*?<authors>.*?</authors>.*?</secondary sources>', output_text, re.DOTALL):
        meta["secondary sources-authors"] = True
    
    return meta

# ---------- Load and process all JSON files ----------
combined_examples = []

# Get all JSON files from first_selected folder
json_files = glob.glob(os.path.join(first_selected_folder, "*.json"))

print(f"Found {len(json_files)} JSON files in {first_selected_folder}")

for json_file in json_files:
    filename = os.path.basename(json_file)
    print(f"\n   Processing: {filename}")
    
    # Load JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        examples = json.load(f)
    
    # Filter for selected examples only
    selected_examples = [ex for ex in examples if ex.get("selected", False)]
    print(f"   → Found {len(selected_examples)} selected examples")
    
    # Add metadata and combine
    for example in selected_examples:
        # Extract meta flags from output
        meta = extract_meta_flags(example["example"]["output"])
        
        # Create new entry with source file and meta
        combined_entry = {
            "example": example["example"],
            "selected": False, # Set to false,
            "rejected": False,
            "notation": example["notation"],
            "comments": example["comments"],
            "source_file": filename,
            "meta": meta
        }
        
        combined_examples.append(combined_entry)

print(f"\n   ✓ Total selected examples collected: {len(combined_examples)}")

# ---------- Save combined results ----------
with open(output_combined_path, 'w', encoding='utf-8') as f:
    json.dump(combined_examples, f, indent=2, ensure_ascii=False)

print(f"   ✓ Saved combined examples to: {output_combined_path}")


Found 7 JSON files in C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\first_selected

   Processing: 1989CanLII1415ONCA.json
   → Found 35 selected examples

   Processing: 1997CanLII16226.json
   → Found 155 selected examples

   Processing: 2001CanLII21117QCTDP.json
   → Found 156 selected examples

   Processing: 2019SCC65_bis.json
   → Found 304 selected examples

   Processing: 2021QCCA1675.json
   → Found 21 selected examples

   Processing: 2024NBKB203.json
   → Found 116 selected examples

   Processing: combined_selected_examples.json
   → Found 0 selected examples

   ✓ Total selected examples collected: 787
   ✓ Saved combined examples to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\first_selected\combined_selected_examples.json


In [4]:
# ---------- Display sample of combined examples ----------
print(f"Total examples: {len(combined_examples)}\n")
print("Sample entry:")
print(json.dumps(combined_examples[0], indent=2, ensure_ascii=False))

# ---------- Display meta statistics ----------
print("\n" + "="*60)
print("Meta Label Statistics:")
print("="*60)

meta_counts = {}
for ex in combined_examples:
    for key, value in ex["meta"].items():
        if key not in meta_counts:
            meta_counts[key] = 0
        if value:
            meta_counts[key] += 1

for label, count in sorted(meta_counts.items()):
    percentage = (count / len(combined_examples)) * 100
    print(f"{label:35s}: {count:3d} examples ({percentage:5.1f}%)")

Total examples: 787

Sample entry:
{
  "example": {
    "input": " decision of the\nMinister of National Revenue dated July 10, 1987, with respect to a notice of\nobjection filed pursuant to section 51.17 of the Excise Tax Act. BETWEEN CAN\nTRAFFIC SERVICES LTD. Appellant AND THE\nMINISTER OF NATIONAL REVENUE Respondent DECISION\nOF THE",
    "output": " decision of the\nMinister of National Revenue dated July 10, 1987, with respect to a notice of\nobjection filed pursuant to <legislation><fragment>section 51.17</fragment> of the <title>Excise Tax Act</title></legislation>. BETWEEN CAN\nTRAFFIC SERVICES LTD. Appellant AND THE\nMINISTER OF NATIONAL REVENUE Respondent DECISION\nOF THE"
  },
  "selected": false,
  "rejected": false,
  "notation": 0,
  "comments": "",
  "source_file": "1989CanLII1415ONCA.json",
  "meta": {
    "legislation": true,
    "legislation-title": true,
    "legislation-fragment": true,
    "legislation-citation": false,
    "decision": false,
    "decision-title":

# Recover Source File Information for combined_v3

In [3]:
import json
import os
import glob

# ---------- Configuration ----------
original_folder = fr"{project_root}\few_shot_selection_tool\original"
combined_v3_path = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3.json"
output_path = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources.json"

# ---------- Load all original files ----------
print("Loading original files...")
original_examples = {}

json_files = glob.glob(os.path.join(original_folder, "*.json"))
print(f"Found {len(json_files)} JSON files in original folder\n")

for json_file in json_files:
    filename = os.path.basename(json_file)
    print(f"   Loading: {filename}")
    
    with open(json_file, 'r', encoding='utf-8') as f:
        examples = json.load(f)
    
    # Store each example with its source file
    for ex in examples:
        # Create a unique key based on example content
        if "example" in ex:
            example_key = (ex["example"]["input"], ex["example"]["output"])
            original_examples[example_key] = filename
    
    print(f"      → Loaded {len(examples)} examples")

print(f"\n✓ Total unique examples indexed: {len(original_examples)}")

# ---------- Load combined_v3 ----------
print(f"\nLoading combined_v3.json...")
with open(combined_v3_path, 'r', encoding='utf-8') as f:
    combined_v3 = json.load(f)

print(f"✓ Loaded {len(combined_v3)} examples from combined_v3.json")

# ---------- Match and update source files ----------
print("\nMatching examples to source files...")
matched = 0
unmatched = 0

for ex in combined_v3:
    example_key = (ex["example"]["input"], ex["example"]["output"])
    
    if example_key in original_examples:
        ex["source_file"] = original_examples[example_key]
        matched += 1
    else:
        ex["source_file"] = "UNKNOWN"
        unmatched += 1

print(f"   ✓ Matched: {matched} examples")
if unmatched > 0:
    print(f"   ⚠ Unmatched: {unmatched} examples")

# ---------- Save updated combined_v3 ----------
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(combined_v3, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved updated file to: {output_path}")
print(f"   Original file preserved at: {combined_v3_path}")

Loading original files...
Found 4 JSON files in original folder

   Loading: few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json
      → Loaded 95 examples
   Loading: few_shot_examples_1997CanLII16226_ONCA_annotated_EG_tech.json
      → Loaded 852 examples
   Loading: few_shot_examples_2019SCC65_annotated_EG_tech_corrected.json
      → Loaded 278 examples
   Loading: few_shot_examples_2021QCCA1675_annotated_EG_tech.json
      → Loaded 90 examples

✓ Total unique examples indexed: 1315

Loading combined_v3.json...
✓ Loaded 270 examples from combined_v3.json

Matching examples to source files...
   ✓ Matched: 270 examples

✓ Saved updated file to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3_with_sources.json
   Original file preserved at: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3.json


# Fix Spacing Issues in Tags

In [4]:
import json
import re

# ---------- Configuration ----------
input_file = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources.json"
output_file = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json"

input_file = fr"{project_root}\few_shot_selection_tool\first_selected\combined_selected_examples.json"
output_file = fr"{project_root}\few_shot_selection_tool\first_selected\combined_selected_examples_fixed_spacing.json"

def fix_tag_spacing(text):
    """
    Fix spacing in tags: move spaces after opening tags to before opening tags.
    Example: '<tag> word</tag>' -> ' <tag>word</tag>
    A mention never starts with a space
    """
    import re
    
    # Tokenize: match tags, whitespace sequences, and text chunks
    # Tags: <...>, Whitespace: \s+, Text: sequences without < or whitespace
    pattern = r'<[^>]+>|\s+|[^<\s]+'
    tokens = re.findall(pattern, text)
    
    # Swap adjacent pairs where opening tag is followed by whitespace
    i = 0
    while i < len(tokens) - 1:
        current, next_tok = tokens[i], tokens[i + 1]
        
        # Check if current is opening tag (starts with <, not </)
        # and next token is whitespace
        is_opening_tag = current.startswith('<') and not current.startswith('</')
        is_whitespace = next_tok.isspace()
        
        if is_opening_tag and is_whitespace:
            # Swap the two tokens: space moves before tag
            tokens[i], tokens[i + 1] = next_tok, current
            i += 2  # Skip both tokens since we just processed them
        else:
            i += 1
    
    # Reconstruct and return the corrected text
    return ''.join(tokens)

# ---------- Load the file ----------
print(f"Loading: {input_file}")
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✓ Loaded {len(data)} examples\n")

# ---------- Fix spacing in all examples ----------
print("Fixing spacing issues...")
fixed_count = 0
total_changes = 0

for idx, ex in enumerate(data):
    # Fix spacing in both input and output
    original_input = ex["example"]["input"]
    original_output = ex["example"]["output"]
    
    fixed_input = fix_tag_spacing(original_input)
    fixed_output = fix_tag_spacing(original_output)
    
    # Track changes
    if original_input != fixed_input or original_output != fixed_output:
        fixed_count += 1
        if original_input != fixed_input:
            total_changes += 1
        if original_output != fixed_output:
            total_changes += 1
    
    ex["example"]["input"] = fixed_input
    ex["example"]["output"] = fixed_output

print(f"   ✓ Fixed spacing in {fixed_count} examples ({total_changes} fields changed)")

# ---------- Save corrected file ----------
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved corrected file to: {output_file}")
print(f"   Original file preserved at: {input_file}")

Loading: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\first_selected\combined_selected_examples.json
✓ Loaded 787 examples

Fixing spacing issues...
   ✓ Fixed spacing in 80 examples (80 fields changed)

✓ Saved corrected file to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\first_selected\combined_selected_examples_fixed_spacing.json
   Original file preserved at: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\first_selected\combined_selected_examples.json


# From simplifed to normal form

In [3]:
import json
import re

# Start from the previous output file (fixed spacing)
input_file = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json"
output_file = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources_manual_label.json"

def simplified_to_manual_label(text: str) -> str:
    """
    Convert simplified tags to manual_label format.
    Example:
      <decision>...</decision>
      -> <manual_label labelname="decision">...</manual_label>
    """
    # Tokenize tags vs non-tags
    tokens = re.findall(r"<[^>]+>|[^<]+", text)
    converted = []

    for tok in tokens:
        if tok.startswith("<") and tok.endswith(">"):
            # Keep already-normal tags unchanged
            if tok.startswith("<manual_label") or tok == "</manual_label>":
                converted.append(tok)
                continue

            # Closing tag: </xxx> -> </manual_label>
            m_close = re.fullmatch(r"</\s*([^>\s]+(?:\s+[^>\s]+)*)\s*>", tok)
            if m_close:
                converted.append("</manual_label>")
                continue

            # Opening tag: <xxx> -> <manual_label labelname="xxx">
            m_open = re.fullmatch(r"<\s*([^/\s>]+(?:\s+[^>\s]+)*)\s*>", tok)
            if m_open:
                label = m_open.group(1).strip()
                converted.append(f'<manual_label labelname="{label}">')
                continue

            # Fallback
            converted.append(tok)
        else:
            converted.append(tok)

    return "".join(converted)

print(f"Loading: {input_file}")
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

changed = 0
for ex in data:
    original_output = ex["example"]["output"]
    converted_output = simplified_to_manual_label(original_output)
    if converted_output != original_output:
        changed += 1
    ex["example"]["output"] = converted_output  # only output, as requested

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Converted output tags in {changed} examples")
print(f"✓ Saved file: {output_file}")


Loading: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json
✓ Converted output tags in 270 examples
✓ Saved file: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3_with_sources_manual_label.json


# Clean with only selected examples

In [5]:
import json

# ---------- Configuration ----------
input_file = fr"{project_root}\few_shot_selection_tool\second_selected\examples_selected_45.json"
output_file = fr"{project_root}\few_shot_selection_tool\second_selected\examples_selected_45_clean.json"

# ---------- Load and filter examples ----------
print(f"Loading: {input_file}")
with open(input_file, 'r', encoding='utf-8') as f:
    all_examples = json.load(f)

# Filter for selected examples only
selected_examples = [ex for ex in all_examples if ex.get("selected", False)]

print(f"   ✓ Loaded {len(all_examples)} total examples")
print(f"   ✓ Filtered {len(selected_examples)} selected examples")

# ---------- Save selected examples only ----------
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(selected_examples, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved selected examples to: {output_file}")
print(f"   Original file preserved at: {input_file}")

Loading: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\examples_selected_45.json
   ✓ Loaded 787 total examples
   ✓ Filtered 45 selected examples

✓ Saved selected examples to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\examples_selected_45_clean.json
   Original file preserved at: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\examples_selected_45.json


In [7]:
for example in selected_examples:
    print(example["example"]["output"])


 Peter
C. Engelmann, for the respondent Statutes Cited: <legislation> <title>Canadian
International Trade Tribunal Act</title>, <citation>S.C. 1988, c. 56</citation>,<fragment>subs.54(2)</fragment> and <fragment>s. 60</fragment></legislation>; <legislation><title>Excise
Tax Act</title>, <citation>R.S.C. 1970, c. E-13</citation>, <fragment>paragraph 1(h), Part XII, Schedule
III</fragment></legislation>. Cases

Cited: <secondary sources><title>Funk
&amp; Wagnalls New Standard Dictionary of the English Language</title></secondary sources>; <secondary sources><title>The Oxford
English Dictionary</title> (<source>Second Edition</source>)</secondary sources>; <secondary sources><title>Black's Law Dictionary </title>(<source>Revised
Fourth Edition</source>)</secondary sources>; <secondary sources><title>The Houghton Mifflin Canadian Dictionary of the English
Language</title></secondary sources>; <secondary sources><title>Webster's Third New International Dictionary of the English Language
Una